In [1]:
import pandas as pd

df = pd.read_excel("units/lushfoil_platform.xlsx", skiprows=2)

In [2]:
folder_path = "wishlists/Annapurna Interactive Historical Revenues - releases_dates.csv"
dates  = pd.read_csv(folder_path)
dates = dates.dropna(subset='pc_release_date')
dates['pc_release_date'] = pd.to_datetime(dates['pc_release_date'])
release_date_dict = dates.groupby('product')['pc_release_date'].min().to_dict()

In [3]:
release_date_dict.keys()

dict_keys(['A Memoir Blue', 'Ashen', 'BattleSage', 'Cocoon', 'Donut County', 'Due Process', 'Faraway', 'Flock', 'Florence', 'Flower', 'Gorogoa', 'Hindsight', 'Hohokum', 'I Am Dead', 'If Found...', 'Journey', 'Kentucky Route Zero: TV Edition', 'Last Stop', 'Lorelei and the Laser Eyes', 'Lushfoil Photography Sim', 'Maquette', 'Mixtape', 'Morsels', 'Mundaun', 'Neon White', 'Open Roads', 'Outer Wilds', 'Sayonara Wild Hearts', 'Skin Deep', 'Solar Ash', 'Storyteller', 'Stray', 'Telling Lies', 'The Artful Escape', 'The Lost Wild', 'The Pathless', 'The Unfinished Swan', 'Thirsty Suitors', 'Twelve Minutes', 'Wanderstop', 'Wattam', 'What Remains of Edith Finch', 'Wheel World', 'to a T'])

In [4]:
TITLE =  'Lushfoil Photography Sim'

In [5]:
import glob
import os

# Define folder path
folder_path = "/Users/dougs/Documents/GitHub/IndieBI-Sales-EDA/units/"

# Get all .xlsx files in the folder
files = glob.glob(f"{folder_path}/*.csv")

# Read all files into DataFrames and add filename column
dfs = [pd.read_csv(file).assign(Source=os.path.basename(file)) for file in files]
av_df = pd.concat(dfs, ignore_index=True)
av_df['platform'] = av_df['platform'].str.title()
av_df['platform'] = av_df['platform'].replace({'Playstation': 'PlayStation'})
av_df['platform'] = av_df['platform'].replace({'Xbox': 'Microsoft'})


In [6]:
curve_df = av_df.pivot(columns='platform', index='Weeks_from_Release', values="daily_delta").reset_index()

In [7]:
platform_list = list(df.portal.unique())

In [8]:
df = df.pivot(columns='portal', index='Day', values="Cumulative Selected Measure").reset_index()

In [9]:
df['release_date'] = release_date_dict.get(TITLE)
df['release_date'] = pd.to_datetime(df['release_date'])

In [10]:
df["DAR"] = (df["Day"] - df["release_date"]).dt.days
df["Weeks_from_Release"] = (df["DAR"] // 7) + 1  # Week 1 starts at 0-7 days


In [11]:
#df = df[['Weeks_from_Release']+platform_list]

In [12]:
actual_df = df.sort_values(by='DAR').drop_duplicates("Weeks_from_Release", keep='last')

In [13]:
actual_df = actual_df.drop(["Day",'release_date'], axis=1)

In [14]:
actual_df

portal,Epic,Microsoft,PlayStation,Steam,DAR,Weeks_from_Release
6,95,2382,3297,19474,6,1
13,142,3071,4185,25080,13,2
20,152,3281,4585,26457,20,3
27,193,3419,4807,27867,27,4
34,234,3515,4966,28699,34,5
41,238,3592,5097,29354,41,6
48,244,3667,5179,29994,48,7
55,252,3733,5331,30571,55,8
57,252,3733,5339,30636,57,9


In [15]:
curve_df

platform,Weeks_from_Release,Apple,Epic,Gog,Google,Humble,Microsoft,Nintendo,PlayStation,Steam
0,1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2,1.023745,0.748859,0.267076,0.620922,0.576177,0.225106,1.081480,0.541367,0.426685
2,3,0.428988,1.243095,1.529236,0.229504,0.243324,0.289294,0.294547,0.503304,0.339322
3,4,0.349177,0.265075,0.127330,0.249465,0.044030,0.182564,0.124093,0.238531,0.297499
4,5,0.208720,0.148317,0.022225,0.184109,0.015207,0.117121,0.084706,0.261855,0.209352
...,...,...,...,...,...,...,...,...,...,...
151,152,NaN,0.004581,0.001176,0.002021,0.000300,0.000843,0.007436,0.004844,0.002589
152,153,NaN,0.002497,0.001029,0.008588,0.017527,0.000812,0.001912,0.005275,0.004008
153,154,NaN,0.006628,0.001299,0.010736,0.030523,0.000764,0.003941,0.004172,0.003899
154,155,NaN,0.015940,0.004310,0.002687,0.030812,0.000738,0.005332,0.001879,0.004603


In [16]:
import pandas as pd
import numpy as np
actual_df = actual_df.set_index('Weeks_from_Release')
curve_df = curve_df.set_index('Weeks_from_Release')


# 2. Get the last actual week and values
last_week = actual_df.index.max()
last_values = actual_df.loc[last_week].copy()

# 3. Create a DataFrame to hold projections
projected_df = pd.DataFrame()

# 4. Iteratively apply the growth deltas week by week
current_values = last_values.copy()

for week in range(last_week + 1, curve_df.index.max() + 1):
    if week not in curve_df.index:
        continue
    deltas = curve_df.loc[week].fillna(0)  # Default to 0 growth if missing
    current_values = current_values * (1 + deltas)
    current_values.name = week
    current_values = np.ceil(current_values.fillna(0)).astype(int)
    projected_df = pd.concat([projected_df, current_values.to_frame().T])

# 5. Combine with actuals
full_df = pd.concat([actual_df, projected_df])

# 6. Reset index if needed
full_df = full_df.reset_index()

In [17]:
full_df = full_df.drop("index", axis=1)

In [18]:
filtered_df = full_df[full_df.index < 52]
filtered_df

,Epic,Microsoft,PlayStation,Steam,DAR,Apple,Gog,Google,Humble,Nintendo
0,95,2382,3297,19474,6,NaN,NaN,NaN,NaN,NaN
1,142,3071,4185,25080,13,NaN,NaN,NaN,NaN,NaN
2,152,3281,4585,26457,20,NaN,NaN,NaN,NaN,NaN
3,193,3419,4807,27867,27,NaN,NaN,NaN,NaN,NaN
4,234,3515,4966,28699,34,NaN,NaN,NaN,NaN,NaN
5,238,3592,5097,29354,41,NaN,NaN,NaN,NaN,NaN
6,244,3667,5179,29994,48,NaN,NaN,NaN,NaN,NaN
7,252,3733,5331,30571,55,NaN,NaN,NaN,NaN,NaN
8,252,3733,5339,30636,57,NaN,NaN,NaN,NaN,NaN
9,262,4286,5645,32998,0,0.0,0.0,0.0,0.0,0.0


In [19]:
full_df.loc[51]

Epic            1044.0
Microsoft      24415.0
PlayStation    14501.0
Steam          88703.0
DAR                0.0
Apple              0.0
Gog                0.0
Google             0.0
Humble             0.0
Nintendo           0.0
Name: 51, dtype: float64

In [20]:
full_df.loc[51].sum()

128663.0

In [21]:
full_df.loc[103].sum()

224514.0

In [22]:
full_df.loc[155].sum()

312755.0

In [23]:
curve_df.query("Weeks_from_Release <=52").to_clipboard()